# Tokenization & Context

**Prerequisites:** `01_your_first_llm.ipynb`, `02_conversations.ipynb`

Notebook 2's `messages` list grows every turn. This notebook explains what that growth actually costs you: tokens, a hard context limit, real money, and latency. Every number here is measured against the live Groq API and its published pricing, not textbook estimates.

**By the end you'll have:**
- Seen text actually split into tokens
- Proven that your own token estimate and the API's real bill can differ
- Triggered a real context-window error
- Built `trim_to_budget()` and `Conversation.trim()` in `chat.py`
- Computed the real dollar cost and measured real latency of a call

In [1]:
import sys
import time
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import tiktoken

from llm import client, DEFAULT_MODEL
from chat import Conversation, count_tokens, trim_to_budget

## Tokenization

A model doesn't see words, it sees tokens — sub-word chunks from a fixed vocabulary. We'll use `tiktoken` (OpenAI's tokenizer) purely to *illustrate* the concept, because it needs no download or auth. It is **not** the tokenizer Llama 3 (the model behind `llama-3.3-70b-versatile`) actually uses — the exact split would differ — but the underlying idea, breaking text into reusable sub-word pieces, is the same for any modern LLM tokenizer.

In [2]:
enc = tiktoken.get_encoding("cl100k_base")

text = "Tokenization splits text into chunks a model can process."
tokens = enc.encode(text)

print("token count:", len(tokens))
print("pieces:", [enc.decode([t]) for t in tokens])

token count: 11
pieces: ['Token', 'ization', ' splits', ' text', ' into', ' chunks', ' a', ' model', ' can', ' process', '.']


Notice "Tokenization" itself splits into `Token` + `ization` — common words are often one token, rarer or compound words get split into pieces. That's why token count tracks roughly, but not exactly, with word count.

## What actually gets billed: real vs. estimated

`tiktoken`'s count is our estimate. The API's response carries the real number in `usage`. Let's compare them for the exact same text.

In [3]:
response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": text}],
)

print("tiktoken estimate (raw text only):", count_tokens(text))
print("real prompt_tokens (from the API):", response.usage.prompt_tokens)

tiktoken estimate (raw text only): 11
real prompt_tokens (from the API): 46


The real number is noticeably higher. Two reasons: `tiktoken` is the wrong vocabulary for this model, and — more importantly — it only counts your raw text. The API additionally counts the chat template wrapped around every message (role markers, message boundaries) that the model actually sees. `usage.prompt_tokens` is always ground truth; a local estimate is only ever a budgeting approximation.

## Context window

`llama-3.3-70b-versatile` has a **131,072 token** (~128k) context window — the hard ceiling on `prompt_tokens + completion_tokens` combined, per Groq's published model specs. Exceed it and the call fails outright. Let's actually trigger that:

In [4]:
huge_text = "test " * 140_000  # well past 131,072 tokens

try:
    client.chat.completions.create(
        model=DEFAULT_MODEL,
        messages=[{"role": "user", "content": huge_text}],
    )
except Exception as e:
    print(type(e).__name__, ":", e)

BadRequestError : Error code: 400 - {'error': {'message': 'Please reduce the length of the messages or completion.', 'type': 'invalid_request_error', 'param': 'messages'}}


That's a real `400 Bad Request` from Groq, not a client-side guard. This is why long-running agents and chatbots can't just let `messages` grow forever.

## Prompt truncation

The fix: before every call, check whether history fits a token budget, and if not, drop the oldest turns. The system message stays (it defines the assistant's behavior for the whole conversation), and turns are dropped in **user+assistant pairs** — dropping only half a pair would leave a reply with no question behind it.

There's a second problem beyond just capping the *current* history, though: a budget that only accounts for what's already there leaves nothing for the next turn or the reply about to come back. `trim_to_budget()` handles both by reserving room for a few *future* exchanges upfront — `reserve_turns` sets how many, and the size to reserve per turn isn't guessed as a fixed constant, it's computed from the conversation's own history: the average token count of the turns already there. ("Turn" rather than "pair" — here they're the same thing, but a turn won't always be exactly two messages once tool calls enter the picture, from notebook 9 onward.)

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "My favorite color is teal."},
    {"role": "assistant", "content": "Got it, teal is a great choice!"},
    {"role": "user", "content": "I also love hiking on weekends."},
    {"role": "assistant", "content": "Hiking is a wonderful way to enjoy nature."},
    {"role": "user", "content": "What do you know about me?"},
]

print("total tokens:", sum(count_tokens(m["content"]) for m in messages))
print()

for max_tokens, reserve_turns in [(60, 0), (60, 2), (90, 2)]:
    trimmed = trim_to_budget(messages, max_tokens=max_tokens, reserve_turns=reserve_turns)
    kept_tokens = sum(count_tokens(m["content"]) for m in trimmed)
    print(f"max_tokens={max_tokens}, reserve_turns={reserve_turns}: kept {len(trimmed)} messages ({kept_tokens} tokens)")
    for m in trimmed:
        print("  -", m["role"], ":", m["content"])
    print()

All three runs start from the same 45-token conversation, made of 3 turns: `teal` (15 tokens), `hiking` (17 tokens), and a trailing, not-yet-answered `"What do you know about me?"` (7 tokens, a turn of just one message). `reserve_turns=0` at `max_tokens=60` reserves nothing, so it keeps everything, exactly like a plain token cap would. `reserve_turns=2` at the same `max_tokens=60` computes `avg_turn_tokens` across all 3 turns — `(15 + 17 + 7) / 3 ≈ 13` — reserves `2 × 13 ≈ 26` tokens for two future exchanges, and trims the real usable budget down to `60 − 26 = 34`. That's enough to drop only the oldest turn (`teal`, 15 tokens) and keep the rest: 4 messages, 30 tokens. Raise `max_tokens` to `90` with the same `reserve_turns=2`, though, and the reservation (still ~26 tokens) leaves a budget of `64` — comfortably above the 45 tokens actually in use, so nothing gets trimmed at all. Reserving room doesn't mean *always* trimming; it only kicks in once the reservation plus the real history would exceed what's available.

That's `trim_to_budget()` in `chat.py`, and `Conversation.trim()` calls it on its own history:

```python
def _role(m):
    return getattr(m, "role", None) or m.get("role")


def _content(m):
    return getattr(m, "content", None) if hasattr(m, "content") else m.get("content")


def _tokens(m):
    content = _content(m)
    return count_tokens(content) if content else 0


def trim_to_budget(messages, max_tokens, reserve_turns=2):
    system = [m for m in messages if _role(m) == "system"]
    rest = [m for m in messages if _role(m) != "system"]

    turns = []
    for m in rest:
        if _role(m) == "user" or not turns:
            turns.append([m])
        else:
            turns[-1].append(m)

    num_turns = len(turns)
    total_tokens = sum(_tokens(m) for turn in turns for m in turn)
    avg_turn_tokens = total_tokens / num_turns if num_turns else 0
    history_budget = max_tokens - reserve_turns * avg_turn_tokens

    def current_tokens():
        return sum(_tokens(m) for m in system) + sum(_tokens(m) for turn in turns for m in turn)

    while turns and current_tokens() > history_budget:
        turns.pop(0)

    return system + [m for turn in turns for m in turn]


class Conversation:
    # __init__ and send() are from notebook 2 — this is the one method added here
    def trim(self, max_tokens, reserve_turns=2):
        self.messages = trim_to_budget(self.messages, max_tokens, reserve_turns)
```

A "turn" groups a `user` message with everything that follows until the next `user` message — for a plain text conversation like this one, that's always exactly one `user` + one `assistant` message, so it behaves identically to counting pairs. `_role()`/`_content()` look like unnecessary indirection for that simple case — plain dicts already have `["role"]` and `["content"]` — but they exist so this same function keeps working once messages aren't all plain dicts anymore. That happens starting notebook 9: a tool call gets appended as a real SDK message object, not a dict, and its `content` is `None` while a tool call is pending. `avg_turn_tokens` is computed once, from whatever turns already exist — a self-calibrating estimate of "how big does a typical exchange here tend to be," instead of a fixed guess. With no history yet to measure from, `avg_turn_tokens` is `0` — nothing to reserve for, and nothing trims.

In [6]:
convo = Conversation(system="You are a concise assistant. Answer in one sentence.")
convo.send("My favorite color is teal.")
convo.send("I also love hiking on weekends.")
convo.send("My dog's name is Max.")

print("before trim:", len(convo.messages), "messages")
convo.trim(max_tokens=130)
print("after trim: ", len(convo.messages), "messages")
for m in convo.messages:
    print("  -", m["role"], ":", m["content"])

before trim: 7 messages
after trim:  3 messages
  - system : You are a concise assistant. Answer in one sentence.
  - user : My dog's name is Max.
  - assistant : Max is likely a loving and loyal companion, bringing joy and excitement to your life with his energetic and affectionate personality.


Three real exchanges happened, but only the newest survives. With `reserve_turns` defaulting to `2`, the budget sets aside room for two *future* exchanges on top of whatever's already there — so at a modest 130-token limit, keeping more than the single most recent turn would leave no headroom for what's coming next.

## Cost

Groq bills `llama-3.3-70b-versatile` at **$0.59 per million input tokens** and **$0.79 per million output tokens** (from Groq's published pricing). Real `usage` numbers and real prices give a real cost, not a guess:

In [7]:
INPUT_PRICE_PER_TOKEN = 0.59 / 1_000_000
OUTPUT_PRICE_PER_TOKEN = 0.79 / 1_000_000

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Write a 200 word story about a robot."}],
    max_tokens=300,
)

cost = (
    response.usage.prompt_tokens * INPUT_PRICE_PER_TOKEN
    + response.usage.completion_tokens * OUTPUT_PRICE_PER_TOKEN
)
print(f"prompt_tokens={response.usage.prompt_tokens}, completion_tokens={response.usage.completion_tokens}")
print(f"cost of this one call: ${cost:.6f}")

prompt_tokens=45, completion_tokens=239
cost of this one call: $0.000215


That single call costs a fraction of a cent — but multiply by thousands of calls, or by a `messages` list that resends the same growing history every turn (notebook 2), and cost scales with tokens sent, not just tokens you typed.

## Latency

Latency isn't just network time — it also depends on how many tokens the model has to generate. Same prompt-call pattern, short vs. long completion:

In [8]:
start = time.perf_counter()
short = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Say hi."}],
    max_tokens=5,
)
short_latency = time.perf_counter() - start

start = time.perf_counter()
long = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": "Write a 200 word story about a robot."}],
    max_tokens=300,
)
long_latency = time.perf_counter() - start

print(f"short reply ({short.usage.completion_tokens} tokens): {short_latency:.2f}s")
print(f"long reply  ({long.usage.completion_tokens} tokens): {long_latency:.2f}s")

short reply (5 tokens): 0.35s
long reply  (243 tokens): 1.02s


## What you built

✓ Watched text actually split into tokens

✓ Measured the real gap between an estimated token count and the API's actual bill

✓ Triggered a real context-window error at 131,072 tokens

✓ Built `trim_to_budget()` and `Conversation.trim()` in `chat.py` — reserving room for `reserve_turns` future exchanges upfront, sized from the conversation's own history (`avg_turn_tokens`) instead of a guessed constant

✓ Computed a real dollar cost from real `usage` data and real Groq pricing

✓ Measured real latency and saw why it doesn't scale linearly with token count

**Next:** `04_prompting.ipynb` — instructions, few-shot examples, delimiters, and prompt templates.